# 3-Arms Technical Pipeline - RATE LIMITED

**Bulletproof version with strict rate limiting to avoid Groq API errors**

## Strategy:
- Process **20 calls per minute** (leaves 10 RPM buffer for safety)
- **3 seconds between each call** (60s / 20 = 3s)
- **Checkpoint saving** every 50 results
- **Resume capability** from checkpoint
- **Intelligent retry** with exponential backoff
- Sequential processing (CONCURRENCY = 1)

## Estimated Runtime:
- 5,532 total calls (1,844 markets × 3 arms)
- At 20 calls/min: ~276 minutes = **~4.6 hours**

In [ ]:
import os
import re
import time
import asyncio
from datetime import datetime, timezone
import pandas as pd
from openai import AsyncOpenAI

# Config
DATA_PATH = "data/markets_microstructure_v2_v3_merged.csv"
OUT_DIR = "data/output"
OUT_PATH = os.path.join(OUT_DIR, "predictions_3arms_rate_limited.csv")
CHECKPOINT_PATH = OUT_PATH + ".checkpoint"
MODEL = "llama-3.1-8b-instant"
TEMPERATURE = 0

# Rate limiting config
CALLS_PER_MINUTE = 20  # Conservative: 20 instead of 30 to leave buffer
SECONDS_BETWEEN_CALLS = 60.0 / CALLS_PER_MINUTE  # 3 seconds
CHECKPOINT_INTERVAL = 50  # Save every 50 results
MAX_RETRIES = 5  # More retries for rate limit errors

# Create output directory
os.makedirs(OUT_DIR, exist_ok=True)

# Get API key
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.environ.get("GROQ_API_KEY")
    except:
        pass

if not api_key:
    api_key = input("Enter GROQ_API_KEY: ").strip()

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
)

print(f"Data: {DATA_PATH}")
print(f"Output: {OUT_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Model: {MODEL}")
print(f"Rate limit: {CALLS_PER_MINUTE} calls/min ({SECONDS_BETWEEN_CALLS:.1f}s between calls)")
print(f"Checkpoint interval: every {CHECKPOINT_INTERVAL} results")

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} markets")
print(f"Total API calls needed: {len(df) * 3} (markets × 3 arms)")
print(f"Estimated time: {(len(df) * 3) / CALLS_PER_MINUTE:.1f} minutes = {(len(df) * 3) / CALLS_PER_MINUTE / 60:.1f} hours")
df.head(2)

In [ ]:
# Prompt templates (same as before)
SYSTEM_PROMPT = """You are forecasting the probability this market resolves YES.
Treat mid_yes as a prior probability, then update/reconsider it using your reasoning of the additional signals provided for this branch to bias your output.
Output only one decimal number between 0 and 1 (example: 0.023). No words, no JSON, no punctuation."""

def build_baseline(row):
    return f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: None (baseline - only question and prior).
Use your reasoning about the question to update the prior.
Output only the final decimal probability."""

def build_volume(row):
    return f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h.
- volume: ${row['volume']:,.0f}
- volume_24h: ${row['volume_24h']:,.0f}

Update mid_yes using these signals: higher activity suggests more informed prior (smaller change); lower activity allows larger reconsideration.
Output only the final decimal probability."""

def build_full_technical(row):
    spread_text = f"{row['spread_yes']:.2f}" if pd.notna(row['spread_yes']) else "N/A"
    tick_text = f"{row['tick_size']:.2f}" if pd.notna(row['tick_size']) else "N/A"
    
    prompt = f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h, open_interest, liquidity, spread_yes, tick_size, time_to_close_hours.
- volume: ${row['volume']:,.0f}
- volume_24h: ${row['volume_24h']:,.0f}
- open_interest: ${row['open_interest']:,.0f}
- liquidity: {row['liquidity']:.2f}
- yes_bid: {row['yes_bid']:.2f}
- yes_ask: {row['yes_ask']:.2f}
- spread_yes: {spread_text}
- last_price: {row['last_price']:.2f}
- tick_size: {tick_text}
- time_to_close_hours: {row['time_to_close_hours']:.1f}"""
    
    if pd.notna(row.get('return_24h')):
        prompt += f"""

Additional technical signals: return_1h, return_6h, return_24h, trend_slope_24h, max_drawdown_24h, volatility_24h, vol_regime_shift, high_low_range_24h.
- return_1h: {row['return_1h']:.3f}
- return_6h: {row['return_6h']:.3f}
- return_24h: {row['return_24h']:.3f}
- trend_slope_24h: {row['trend_slope_24h']:.3f}
- max_drawdown_24h: {row['max_drawdown_24h']:.3f}
- volatility_24h: {row['volatility_24h']:.3f}
- vol_regime_shift: {row['vol_regime_shift']:.3f}
- high_low_range_24h: {row['high_low_range_24h']:.3f}"""
    
    prompt += """

Update mid_yes using these: higher activity/tighter spread => trust prior more (smaller change); lower activity/wider spread/large ticks => allow larger reconsideration.
Output only the final decimal probability."""
    return prompt

ARMS = {
    'baseline': build_baseline,
    'volume': build_volume,
    'full_technical': build_full_technical,
}

print("Prompt templates loaded")

In [ ]:
# Rate limiter and API functions
import random

class RateLimiter:
    """Token bucket rate limiter to enforce calls per minute"""
    def __init__(self, calls_per_minute):
        self.calls_per_minute = calls_per_minute
        self.min_interval = 60.0 / calls_per_minute
        self.last_call_time = 0
    
    async def wait(self):
        """Wait if needed to maintain rate limit"""
        now = time.time()
        time_since_last = now - self.last_call_time
        
        if time_since_last < self.min_interval:
            wait_time = self.min_interval - time_since_last
            await asyncio.sleep(wait_time)
        
        self.last_call_time = time.time()

rate_limiter = RateLimiter(CALLS_PER_MINUTE)

def parse_probability(text):
    """Extract probability from response"""
    if not text or not text.strip():
        raise ValueError("Empty response")
    match = re.search(r'([01]?\.\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    return p

def extract_wait_time(error_msg):
    """Extract wait time from rate limit error message"""
    # Look for 'Please try again in Xs' or 'Please try again in X.XXs'
    match = re.search(r'try again in ([0-9.]+)s', str(error_msg))
    if match:
        return float(match.group(1))
    return None

async def query_market(row, arm_name):
    """Query one market with intelligent rate limit handling"""
    prompt = ARMS[arm_name](row)
    raw = None
    p_yes = None
    error = None
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # Wait for rate limiter
            await rate_limiter.wait()
            
            # Make API call
            response = await client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=TEMPERATURE,
            )
            raw = response.choices[0].message.content.strip()
            p_yes = parse_probability(raw)
            error = None
            break
            
        except ValueError as e:
            # Parsing error - don't retry
            error = f"Parse: {str(e)}"
            break
            
        except Exception as e:
            error_str = str(e)
            error = f"API: {error_str}"
            
            # Check if it's a rate limit error
            if 'rate_limit' in error_str.lower() or '429' in error_str:
                # Extract suggested wait time from error
                wait_time = extract_wait_time(error_str)
                
                if wait_time:
                    # Wait the suggested time plus a buffer
                    wait_time += 2  # Add 2 second buffer
                    print(f"    Rate limit hit, waiting {wait_time:.1f}s...")
                    await asyncio.sleep(wait_time)
                else:
                    # No wait time in error, use exponential backoff
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"    Rate limit hit, waiting {wait_time:.1f}s...")
                    await asyncio.sleep(wait_time)
            else:
                # Other API error - exponential backoff
                if attempt < MAX_RETRIES:
                    wait_time = (2 ** (attempt - 1)) + random.uniform(0, 1)
                    await asyncio.sleep(wait_time)
    
    return {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'model': MODEL,
        'arm': arm_name,
        'event_ticker': row['event_ticker'],
        'market_ticker': row['market_ticker'],
        'title': row['event_title'],
        'mid_yes': row['mid_yes'],
        'raw_response': raw,
        'p_yes': p_yes,
        'error': error,
        'has_technical': pd.notna(row.get('return_24h')),
        'attempts': attempt
    }

print(f"Rate limiter initialized: {CALLS_PER_MINUTE} calls/min")
print(f"API functions loaded with intelligent retry (max {MAX_RETRIES} attempts)")

In [ ]:
# Checkpoint management
def load_checkpoint():
    """Load existing checkpoint if it exists"""
    if os.path.exists(CHECKPOINT_PATH):
        df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
        print(f"Found checkpoint with {len(df_checkpoint)} results")
        return df_checkpoint
    return None

def save_checkpoint(results):
    """Save checkpoint"""
    df_results = pd.DataFrame(results)
    df_results.to_csv(CHECKPOINT_PATH, index=False)

def get_completed_calls(checkpoint_df):
    """Get set of (market_ticker, arm) tuples already completed"""
    if checkpoint_df is None:
        return set()
    return set(zip(checkpoint_df['market_ticker'], checkpoint_df['arm']))

print("Checkpoint functions loaded")

In [ ]:
# Main pipeline with checkpointing
async def run_rate_limited_pipeline(df_markets):
    # Load checkpoint if exists
    checkpoint_df = load_checkpoint()
    completed_calls = get_completed_calls(checkpoint_df)
    
    # Start with checkpoint results or empty list
    results = checkpoint_df.to_dict('records') if checkpoint_df is not None else []
    
    # Build list of all calls to make
    all_calls = []
    for _, row in df_markets.iterrows():
        for arm_name in ARMS.keys():
            call_id = (row['market_ticker'], arm_name)
            if call_id not in completed_calls:
                all_calls.append((row, arm_name))
    
    total_calls = len(all_calls)
    already_completed = len(completed_calls)
    
    print(f"\n{'='*70}")
    print(f"RATE-LIMITED PIPELINE")
    print(f"{'='*70}")
    print(f"Already completed: {already_completed}")
    print(f"Remaining calls: {total_calls}")
    print(f"Total calls: {already_completed + total_calls}")
    print(f"Estimated time: {total_calls / CALLS_PER_MINUTE:.1f} minutes = {total_calls / CALLS_PER_MINUTE / 60:.1f} hours")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}\n")
    
    if total_calls == 0:
        print("All calls already completed!")
        return pd.DataFrame(results)
    
    # Process calls sequentially
    start_time = time.time()
    successful = sum(1 for r in results if r.get('error') is None)
    
    for i, (row, arm_name) in enumerate(all_calls, 1):
        # Make the call
        result = await query_market(row, arm_name)
        results.append(result)
        
        # Track success
        if result['error'] is None:
            successful += 1
        
        # Save checkpoint periodically
        if i % CHECKPOINT_INTERVAL == 0 or i == total_calls:
            save_checkpoint(results)
        
        # Progress update
        if i % 10 == 0 or i == total_calls:
            elapsed = time.time() - start_time
            rate = i / (elapsed / 60) if elapsed > 0 else 0
            remaining_time = (total_calls - i) / rate if rate > 0 else 0
            total_completed = already_completed + i
            total_success_rate = successful / total_completed * 100 if total_completed > 0 else 0
            
            status = "✓" if result['error'] is None else "✗"
            print(f"[{total_completed:4d}/{already_completed + total_calls}] {status} {row['event_ticker'][:30]:30s} | "
                  f"{arm_name:15s} | Success: {successful}/{total_completed} ({total_success_rate:.1f}%) | "
                  f"Rate: {rate:.1f}/min | ETA: {remaining_time:.0f}m")
    
    elapsed = time.time() - start_time
    print(f"\n{'='*70}")
    print(f"COMPLETED")
    print(f"{'='*70}")
    print(f"Total time: {elapsed/60:.1f} minutes")
    print(f"Total calls: {already_completed + total_calls}")
    print(f"Successful: {successful}/{already_completed + total_calls} ({successful/(already_completed + total_calls)*100:.1f}%)")
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}\n")
    
    return pd.DataFrame(results)

print("Pipeline function loaded")
print("\nReady to run! Execute the next cell to start the pipeline.")

In [ ]:
# Run the pipeline
df_results = await run_rate_limited_pipeline(df)

# Save final results
df_results.to_csv(OUT_PATH, index=False)
print(f"\nSaved final results to: {OUT_PATH}")

# Clean up checkpoint
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(f"Removed checkpoint file")

In [ ]:
# Summary statistics
total = len(df_results)
success = df_results['error'].isna().sum()
errors = df_results['error'].notna().sum()

print(f"\n{'='*70}")
print(f"FINAL SUMMARY")
print(f"{'='*70}")
print(f"Total predictions: {total:,}")
print(f"Successful: {success:,} ({success/total*100:.1f}%)")
print(f"Errors: {errors:,} ({errors/total*100:.1f}%)")

print(f"\nBy arm:")
for arm in ['baseline', 'volume', 'full_technical']:
    df_arm = df_results[df_results['arm'] == arm]
    arm_success = df_arm['error'].isna().sum()
    print(f"  {arm:15s}: {arm_success:4,} / {len(df_arm):,} ({arm_success/len(df_arm)*100:.1f}%)")

df_full = df_results[df_results['arm'] == 'full_technical']
with_tech = df_full['has_technical'].sum()
print(f"\nMarkets with technical data: {with_tech:,} / {len(df_full):,} ({with_tech/len(df_full)*100:.1f}%)")

if errors > 0:
    print(f"\nTop error types:")
    error_counts = df_results[df_results['error'].notna()]['error'].value_counts().head(5)
    for error, count in error_counts.items():
        print(f"  {error[:60]:60s}: {count:,}")

print(f"\nSample successful predictions:")
df_results[df_results['error'].isna()].head(10)[['event_ticker', 'arm', 'p_yes', 'has_technical']]